# ELItis — Tier-list Thumbnail Generator

**Workflow**

| Cell | Name | Re-run when |
|------|------|-------------|
| **0 · Setup** | Install package, define paths and canvas size | Once per session |
| **1 · Import** | Upload zip/CSV → load or build project | First load; re-run additively to merge more CSVs |
| **2 · Images** | Match or upload images for unmatched items | When images are missing or changed |
| **3a · Font** | Project-wide font + auto-size settings | When changing the font |
| **3b · Text** | Project-wide text color, align, position | When changing text style |
| **3c · Overlay** | Project-wide box, outline, shadow | When changing the overlay |
| **3e · Overrides** | Per-item emergency overrides | When Cell 4 shows warnings |
| **4 · Render** | Full-res render → warnings → gallery | After every tune step |
| **5 · Export** | Download all thumbnails as a zip | Once at the end |

**Tips**
- Labels come from CSV; images are matched by filename stem or assigned manually in Cell 2
- If you already have a project `.json` from the desktop app, drop it in the zip — settings load automatically
- Items with no image render as a grey checkerboard and are flagged in the warnings table
- Loop between **3a–3e → 4** until the gallery looks right, then run **5** once

In [ ]:
# ── Cell 0 · Setup ────────────────────────────────────────────────────────────
# Run once per session.  Install the package and define the paths and canvas
# size constants that every other cell reads.
# ─────────────────────────────────────────────────────────────────────────────

import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "git+https://github.com/JonnyDanny/ELitis", "-q"],
    check=True,
)

from pathlib import Path

PROJECT_DIR = Path("/content/project")
OUTPUT_DIR  = Path("/content/output")

CANVAS_W = 1280
CANVAS_H = 720

print(f"ELItis ready.  Canvas: {CANVAS_W}x{CANVAS_H}  |  Project dir: {PROJECT_DIR}")

In [ ]:
# ── Cell 1 · Import ───────────────────────────────────────────────────────────
# First run: upload a zip (images + optional project.json + optional CSVs).
# Re-run additively: upload a CSV-only zip (or bare CSV) to merge more labels.
# ─────────────────────────────────────────────────────────────────────────────

from elitis.notebook.display import import_panel
from elitis.core.data_io import save_project

project = import_panel(
    PROJECT_DIR, OUTPUT_DIR,
    canvas_w=CANVAS_W, canvas_h=CANVAS_H,
    existing_project=locals().get("project"),  # None on first run → full load
)

# Canonical save path shared by Cells 2 and 4 for auto-save.
# Written here so a VM disconnect after import doesn't lose label/origin data.
_PROJECT_JSON = PROJECT_DIR / f"{project.name}.json"
save_project(project, _PROJECT_JSON)
print(f"Project checkpoint: {_PROJECT_JSON}")

In [ ]:
# ── Cell 2 · Images ────────────────────────────────────────────
# Assign images to items that are still unmatched after Cell 1.
# Bulk-upload by filename stem, paste a URL, or give an absolute path.
# Each assignment auto-saves to _PROJECT_JSON so progress survives a disconnect.
# ─────────────────────────────────────────────────────────────────────────────

from elitis.notebook.images import image_panel

image_panel(project, project_dir=PROJECT_DIR / "Images", project_path=_PROJECT_JSON)

In [ ]:
# ── Cell 3a · Font ────────────────────────────────────────────────
# Project-wide font family and auto-size settings.
# Re-run Cell 4 after making changes to see the effect.
# ─────────────────────────────────────────────────────────────────────────────

from elitis.notebook.tune import font_panel

font_panel(project)

In [ ]:
# ── Cell 3b · Text style ──────────────────────────────────────────────────────
# Project-wide text color, opacity, transform, alignment, and position.
# Re-run Cell 4 after making changes to see the effect.
# ─────────────────────────────────────────────────────────────────────────────

from elitis.notebook.tune import text_panel

text_panel(project)

In [ ]:
# ── Cell 3c · Overlay ─────────────────────────────────────────────────────────
# Project-wide box overlay, text outline, and drop shadow.
# Re-run Cell 4 after making changes to see the effect.
# ─────────────────────────────────────────────────────────────────────────────

from elitis.notebook.tune import overlay_panel

overlay_panel(project)

In [ ]:
# ── Cell 3e · Overrides ───────────────────────────────────────────────────────
# Per-item emergency overrides for items with text fit warnings.
# Also accessible manually to fix anything the renderer does not catch.
# Re-run Cell 4 after applying overrides.
# ─────────────────────────────────────────────────────────────────────────────

from elitis.notebook.overrides import override_panel

override_panel(project)

In [ ]:
# ── Cell 4 · Render ───────────────────────────────────────────────────
# Full-resolution render for all items -> warnings table -> gallery.
# Cell 5 zips what this cell writes -- no re-render needed.
# ─────────────────────────────────────────────────────────────────────────────

import shutil
from pathlib import Path

from elitis.core import renderer
from elitis.core.font_manager import FontManager
from elitis.core.data_io import save_project
from elitis.notebook.display import show_warnings, show_gallery

FONTS_DIR = PROJECT_DIR / "Fonts"
font_manager = FontManager(FONTS_DIR if FONTS_DIR.exists() else Path("/nonexistent"))

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True)

print(f"Rendering {len(project.content_items)} items...")

saved, warnings = renderer.render_all(
    project, font_manager, OUTPUT_DIR,
    on_progress=lambda d, t, p: print(f"  [{d}/{t}] {p.name}"),
)
print(f"\n{len(saved)} images written to {OUTPUT_DIR}")

# Persist last_output_path and any tune settings changed since Cell 1.
save_project(project, _PROJECT_JSON)

show_warnings(warnings)
show_gallery(OUTPUT_DIR)

In [ ]:
# ── Cell 5 · Export ───────────────────────────────────────────────────
# Zips everything Cell 4 rendered and starts a browser download.
# Run Cell 4 first -- this cell does not re-render.
# ─────────────────────────────────────────────────────────────────────────────

import io as _io, zipfile
from pathlib import Path
from google.colab import files

images = sorted(OUTPUT_DIR.glob("*"))
if not images:
    raise RuntimeError("No rendered images found -- run Cell 4 first.")

zip_name = f"{project.name}_thumbnails.zip"
zip_path = Path(f"/content/{zip_name}")

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in images:
        zf.write(p, p.name)

print(f"Downloading {len(images)} images as '{zip_name}' ...")
files.download(str(zip_path))